In [1]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms

In [2]:
batch_size = 256

trainset = datasets.MNIST("./dataset/", train=True, transform = transforms.ToTensor(), download = True)
trainloader = torch.utils.data.DataLoader(trainset, batch_size = batch_size, shuffle = True, num_workers = 2)

testset = datasets.MNIST("./dataset/", train=False, transform = transforms.ToTensor(), download =   True)
testloader = torch.utils.data.DataLoader(testset, batch_size = batch_size, shuffle = False, num_workers = 2)

100%|██████████| 9.91M/9.91M [00:01<00:00, 9.56MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 318kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.86MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.21MB/s]


In [3]:
class SAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder_layer_1 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(28 * 28, 500),
            nn.ReLU(),
        )

        self.decoder_layer_1 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(500, 28 * 28),
            nn.Sigmoid(),
        )

        self.encoder_layer_2 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(500, 500),
            nn.ReLU(),
        )

        self.decoder_layer_2 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(500, 500),
            nn.ReLU(),
        )

        self.encoder_layer_3 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(500, 2000),
            nn.ReLU(),
        )

        self.decoder_layer_3 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(2000, 500),
            nn.ReLU(),
        )

        self.encoder_layer_4 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(2000, 10),
            nn.Sigmoid(),
        )

        self.decoder_layer_4 =  nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(10, 2000),
            nn.ReLU(),
        )

    def forward(self, x):
        encoded = self.encoder_layer_4(
                    self.encoder_layer_3(
                        self.encoder_layer_2(
                            self.encoder_layer_1(x)
                        )
                    )
                )

        decoded = self.decoder_layer_1(
                    self.decoder_layer_2(
                        self.decoder_layer_3(
                            self.decoder_layer_4(encoded)
                        )
                    )
                )
        
        return decoded

# SAE Training

## Pretraining
Greedy layer-wise training

In [ ]:
model = SAE()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

pretrain_epochs = 100
train_epochs = 300


# Pretraining

# Pretrain layer 1
loss_function = nn.MSELoss()

optimizer = optim.SGD(
    list(model.encoder_layer_1.parameters()) + 
    list(model.decoder_layer_1.parameters()),
    lr = 0.1, 
    momentum = 0.9
    )

scheduler = optim.lr_scheduler.StepLR(
    optimizer, 
    step_size = 20000, #Amount of iterations the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

model.train()
for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images, start_dim = 1).to(device)
        e1 = model.encoder_layer_1(images)
        e1_reconstructed = model.decoder_layer_1(e1)
        loss = loss_function(e1_reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain 1: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")

# Pretrain layer 2
loss_function = nn.MSELoss()

optimizer = optim.SGD(
    list(model.encoder_layer_2.parameters()) + 
    list(model.decoder_layer_2.parameters()),
    lr = 0.1, 
    momentum = 0.9
    )

scheduler = optim.lr_scheduler.StepLR(
    optimizer, 
    step_size = 20000, #Amount of iterations the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

# Deactivate Dropout because Layer is Frozen
model.encoder_layer_1.eval()

for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images, start_dim = 1).to(device)

        with torch.no_grad():
            e1 = model.encoder_layer_1(images)
        
        e2 = model.encoder_layer_2(e1)
        e1_reconstructed = model.decoder_layer_2(e2)

        loss = loss_function(e1_reconstructed, e1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain 2: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")


# Pretrain layer 3
loss_function = nn.MSELoss()

optimizer = optim.SGD(
    list(model.encoder_layer_3.parameters()) + 
    list(model.decoder_layer_3.parameters()),
    lr = 0.1, 
    momentum = 0.9
    )

scheduler = optim.lr_scheduler.StepLR(
    optimizer, 
    step_size = 20000, #Amount of iterations the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

# Deactivate Dropout because Layer is Frozen
model.encoder_layer_1.eval()
model.encoder_layer_2.eval()

for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images, start_dim = 1).to(device)

        with torch.no_grad():
            e1 = model.encoder_layer_1(images)
            e2 = model.encoder_layer_2(e1)
        
        e3 = model.encoder_layer_3(e2)
        e2_reconstructed = model.decoder_layer_3(e3)

        loss = loss_function(e2_reconstructed, e2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain 3: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")


# Pretrain layer 4
loss_function = nn.MSELoss()

optimizer = optim.SGD(
    list(model.encoder_layer_4.parameters()) + 
    list(model.decoder_layer_4.parameters()),
    lr = 0.1, 
    momentum = 0.9
    )

scheduler = optim.lr_scheduler.StepLR(
    optimizer, 
    step_size = 20000, #Amount of iterations the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

# Deactivate Dropout because Layer is Frozen
model.encoder_layer_1.eval()
model.encoder_layer_2.eval()
model.encoder_layer_3.eval()

for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images, start_dim = 1).to(device)

        with torch.no_grad():
            e1 = model.encoder_layer_1(images)
            e2 = model.encoder_layer_2(e1)
            e3 = model.encoder_layer_3(e2)
        
        e4 = model.encoder_layer_4(e3)
        e3_reconstructed = model.decoder_layer_4(e4)

        loss = loss_function(e3_reconstructed, e3)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain 4: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")

Pretrain 1: Epoch 1/20, Loss: 0.071024
Pretrain 1: Epoch 2/20, Loss: 0.068522
Pretrain 1: Epoch 3/20, Loss: 0.064411
Pretrain 1: Epoch 4/20, Loss: 0.059118
Pretrain 1: Epoch 5/20, Loss: 0.055750
Pretrain 1: Epoch 6/20, Loss: 0.048168
Pretrain 1: Epoch 7/20, Loss: 0.047916
Pretrain 1: Epoch 8/20, Loss: 0.045691
Pretrain 1: Epoch 9/20, Loss: 0.045279
Pretrain 1: Epoch 10/20, Loss: 0.042030
Pretrain 1: Epoch 11/20, Loss: 0.039359
Pretrain 1: Epoch 12/20, Loss: 0.037319
Pretrain 1: Epoch 13/20, Loss: 0.038474
Pretrain 1: Epoch 14/20, Loss: 0.036449
Pretrain 1: Epoch 15/20, Loss: 0.034795
Pretrain 1: Epoch 16/20, Loss: 0.032974
Pretrain 1: Epoch 17/20, Loss: 0.034599
Pretrain 1: Epoch 18/20, Loss: 0.032835
Pretrain 1: Epoch 19/20, Loss: 0.033763
Pretrain 1: Epoch 20/20, Loss: 0.033107
Pretrain 2: Epoch 1/20, Loss: 0.064568
Pretrain 2: Epoch 2/20, Loss: 0.051592
Pretrain 2: Epoch 3/20, Loss: 0.042260
Pretrain 2: Epoch 4/20, Loss: 0.041583
Pretrain 2: Epoch 5/20, Loss: 0.038482
Pretrain 2: Ep

## Finetuning

In [7]:
loss_function = nn.MSELoss()

optimizer = optim.SGD(
    model.parameters(),
    lr = 0.1, 
    momentum = 0.9
    )

scheduler = optim.lr_scheduler.StepLR(
    optimizer, 
    step_size = 20000, #Amount of iterations the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

train_losses = []

model.eval() # deactivate all dropout
for epoch in range(train_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images, start_dim = 1).to(device)
        reconstructed = model(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        train_losses.append(loss.item())

    print(f"Finetune: Epoch {epoch+1}/{train_epochs}, Loss: {loss.item():.6f}")

Finetune: Epoch 1/50, Loss: 0.065349
Finetune: Epoch 2/50, Loss: 0.059239
Finetune: Epoch 3/50, Loss: 0.054021
Finetune: Epoch 4/50, Loss: 0.051224
Finetune: Epoch 5/50, Loss: 0.045529
Finetune: Epoch 6/50, Loss: 0.042390
Finetune: Epoch 7/50, Loss: 0.042705
Finetune: Epoch 8/50, Loss: 0.042081
Finetune: Epoch 9/50, Loss: 0.038422
Finetune: Epoch 10/50, Loss: 0.035612
Finetune: Epoch 11/50, Loss: 0.036652
Finetune: Epoch 12/50, Loss: 0.034963
Finetune: Epoch 13/50, Loss: 0.034069
Finetune: Epoch 14/50, Loss: 0.034895
Finetune: Epoch 15/50, Loss: 0.031808
Finetune: Epoch 16/50, Loss: 0.031681
Finetune: Epoch 17/50, Loss: 0.030260
Finetune: Epoch 18/50, Loss: 0.031837
Finetune: Epoch 19/50, Loss: 0.032391
Finetune: Epoch 20/50, Loss: 0.030591
Finetune: Epoch 21/50, Loss: 0.031946
Finetune: Epoch 22/50, Loss: 0.030532
Finetune: Epoch 23/50, Loss: 0.030744
Finetune: Epoch 24/50, Loss: 0.029180
Finetune: Epoch 25/50, Loss: 0.028705
Finetune: Epoch 26/50, Loss: 0.029251
Finetune: Epoch 27/50

# Clustering